# Fine-Tuning Llama 2 with QLoRA

Fine-tune `NousResearch/Llama-2-7b-chat-hf` on a 1k-sample instruction dataset using **QLoRA** (4-bit quantization + LoRA adapters).

Stack: `transformers` + `peft` + `bitsandbytes` + `trl`

> **GPU required.** An A100/T4 (16GB+) is recommended.

In [ ]:
!pip install -q accelerate peft bitsandbytes transformers trl datasets

## Imports

In [ ]:
import os
import gc
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

## Llama 2 Chat Prompt Template

The chat variant of Llama 2 expects this format:

```
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_message} [/INST] {model_reply} </s>
```

The dataset `mlabonne/guanaco-llama2-1k` is already pre-formatted with this template — no manual prompt engineering needed.

## Configuration

In [ ]:
# ── Model & data ──────────────────────────────────────────────────────────
model_name   = "NousResearch/Llama-2-7b-chat-hf"
dataset_name = "mlabonne/guanaco-llama2-1k"
new_model    = "Llama-2-7b-chat-finetune"

# ── QLoRA ─────────────────────────────────────────────────────────────────
lora_r       = 64    # LoRA rank
lora_alpha   = 16    # LoRA scaling
lora_dropout = 0.1

# ── 4-bit quantization ────────────────────────────────────────────────────
use_4bit              = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type   = "nf4"
use_nested_quant      = False   # double quantization

# ── Training ──────────────────────────────────────────────────────────────
output_dir                  = "./results"
num_train_epochs            = 1
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing      = True
max_grad_norm               = 0.3
learning_rate               = 2e-4
weight_decay                = 0.001
optim                       = "paged_adamw_32bit"
lr_scheduler_type           = "cosine"
max_steps                   = -1
warmup_ratio                = 0.03
group_by_length             = True
logging_steps               = 25

# Automatically use bf16 on Ampere+ GPUs (A100), else fp16
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    bf16 = major >= 8
    fp16 = not bf16
else:
    bf16 = fp16 = False

# ── SFT ───────────────────────────────────────────────────────────────────
max_seq_length = None
packing        = False

## Load Dataset, Model & Train

In [ ]:
dataset = load_dataset(dataset_name, split="train")

# 4-bit quantization config
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# LoRA config
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# SFTConfig combines TrainingArguments + SFT-specific params
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    gradient_checkpointing=gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim=optim,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    save_strategy="epoch",
    report_to="none",
    # SFT-specific
    max_seq_length=max_seq_length,
    packing=packing,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained(new_model)

## TensorBoard

Run the cells below to visualise training loss and learning rate curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs

## Inference

Test the fine-tuned model before merging.

In [ ]:
logging.set_verbosity(logging.CRITICAL)

prompt = "What is a large language model?"
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]["generated_text"])

In [ ]:
# Free VRAM before merging
del model
del pipe
del trainer
gc.collect()
torch.cuda.empty_cache()

## Merge LoRA Weights into Base Model

LoRA adapters are stored separately. To get a single deployable model, reload the base model in fp16 and call `merge_and_unload()`.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Push to Hugging Face Hub

Authenticate then push the merged model and tokenizer.

In [ ]:
!huggingface-cli login

model.push_to_hub(new_model, create_pr=True)
tokenizer.push_to_hub(new_model, create_pr=True)